# 12 · Collapsed read — q vs KV role in the goal DIFF

Goal heads span ~20 layers, so their q- and KV-states compose the goal representation in a distributed
way (earlier goal heads feed what later ones read). This notebook **collapses** the goal heads to read
a single residual `h^L_COL`, removing that feedback, so the causal role of **q** ("what the pivot is
looking for") vs **KV** ("what the task span exposes") can be separated at one depth.

## Two collapse strategies (`COLLAPSE_STRATEGY`)
- **`super_layer` (Design B, default)** — a synthetic super-layer: sum **all** goal heads' OV write when
  each reads `h^L_COL` (real attention, q/k/v incl. Qwen2 biases), and inject that single vector
  `add_collapse` at `L_COL`. The **native** goal heads (and the rest of `ABLATE_SET`) are ablated so they
  don't contribute again — the goal is expressed once, at one depth. Readout verbalizes `add_collapse`
  directly; generation injects it live each step (KV-cache-safe, `h^L_COL` cache accumulated).
  Validated: a head's reconstructed OV matches its real OV at cosine 1.0000.
- **`read_redirect` (Design A)** — the goal heads at `l > L_COL` recompute their own q/k/v from `h^L_COL`
  but still write natively (surgical `q/k/v_proj` slice overwrite); readout reads those (kept) heads.
  Validated: collapsing a head to read `h^{l-1}` is byte-identical to clean.

## Four runs (each: full generation + readout @ pivot + optional OVReader-layerwise)
1. **clean** — no intervention.
2. **collapsed_clean** — collapse + `ABLATE_SET` knocked out.
3. **collapsed KV-transfer** — collapsed DST reads k/v from SRC task span; q native.
4. **collapsed q-transfer** — collapsed DST q patched from SRC; k/v native.

crossed with **`TRANSFER_SRC ∈ {clean, collapsed_clean}`**. Transfer works under **both** strategies:
under `super_layer` the SRC k/v (or pivot q) are sourced into the super-layer's own computation
(`super_collapse_write` / `super_layer_gen`); under `read_redirect` via `headwise_kv_patch` / `proj_patch_hooks`.

In [1]:
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"   # free GPUs (S3b mlps jobs are on 0,1 / 6,7)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append(".."); sys.path.append("../..")

import json, torch, baukit
from contextlib import contextmanager, nullcontext, ExitStack
from rich import print as rich_print
from rich.markup import escape
from rich.panel import Panel
from rich.text import Text

import src.diff.functional as FUNC
from src.diff.functional import *          # cache_src_proj, headwise_kv_patch, heads_per_layer, greedy,
                                            # align_kv_spans, token_span, make_patch_interface, proj_patch_hooks,
                                            # restrict_head_attn, boost_head_ov, pivot_attn, goalscope_readout …
import src.diff.data as DATA
import scripts.evaluation.eval_ov_reader as R
from scripts.evaluation.eval_goalscope import hard_eval, CAUSAL_PREFIX
from scripts.evaluation.eval_collapse_layerwise import resid_all
from src.diff.h1h2_mask import _proj, _apply_rope, _rope_cos_sin, _GoalReaders, _rmsnorm
from src.attention import get_attention_matrices
from src.goals.goalscope import OVReader, head_additive_generate, _build_goalscope_prompt, _build_causality_prompt
from src.tokens import prepare_input

os.chdir(os.path.abspath(os.path.join(os.path.dirname(R.__file__), "..", "..")))
torch.set_grad_enabled(False)

MODEL = "Qwen/Qwen2.5-Coder-32B-Instruct"
FUNC.setup(MODEL)                  # loads model (eager) + head masks; binds P globals (mt, GROUP, GOAL_HEADS, …)
mt = FUNC.mt

2026-09-14 01:14:07 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


In [2]:
# OPTIONAL — use the top-250 goal-head set (patch + readout), which gave near-perfect kv transfer.
import json
_p = "head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json"
GOAL_HEADS_250 = [tuple(h) for h in json.load(open(_p))]

In [3]:
# ── config ───────────────────────────────────────────────────────────────────────────────────
DATATYPE   = "list[str]"; SRC_LANG = "python"; DST_LANG = "python"
MODE       = "faithfulness"     # faithfulness | causality
KV_STUB    = "var_eq"
SEED = 0

# ── collapse ──
COLLAPSE_STRATEGY = "read_redirect"   # "super_layer" (Design B) | "read_redirect" (Design A)
L_COL             = 36              # single layer the goal heads read from (global)
# GOAL_HEADS_ALL    = list(P.GOAL_HEADS)
GOAL_HEADS_ALL    = GOAL_HEADS_250
# super_layer uses ALL goal heads (offline super-layer, no causality limit);
# read_redirect can only collapse heads strictly above L_COL (within-forward causality).
COLLAPSE_HEADS = (GOAL_HEADS_ALL if COLLAPSE_STRATEGY == "super_layer"
                  else [(l, h) for (l, h) in GOAL_HEADS_ALL if l > L_COL])
READOUT_HEADS  = list(COLLAPSE_HEADS)

# ── ablation (knock out ABLATE_SET so it can't feed / re-do the goal) ──
ABLATE_SET_NAME = "from_lcol"           # "from_lcol" (l>=L_COL) | "band_30_57" | "goal_heads"
ABLATION_STRAT  = "blind_task"          # off | blind_task | blind_task_and_progress | sink_to_pivot
KEEP_COLLAPSED  = True                  # read_redirect: never blind the collapsed heads. super_layer ignores this
LAST_K_KEEP     = 3

# ── transfer (read_redirect only for now) ──
PATCH_HEADS  = list(COLLAPSE_HEADS)
TRANSFER_SRC = "collapsed_clean"                  # "clean" | "collapsed_clean"
OV_BOOST     = 1.0

# ── generation / OVReader ──
GEN_ON = True; GEN_NEW = 96
OV_CLEAN = False; OV_COLLAPSED = True; OV_KV = True; OV_QT = True
LAYERS = list(range(38, 60, 2))
STEER_BAND = (35, 46)
ALPHA_OV = 0.15
MAXNEW = 64

# ── derived ──
def _ablate_set():
    if ABLATE_SET_NAME == "goal_heads": return list(FUNC.GOAL_HEADS)
    if ABLATE_SET_NAME == "from_lcol":  return [(l, h) for l in range(L_COL, mt.n_layer) for h in range(mt.n_heads)]
    if ABLATE_SET_NAME == "band_30_60": return [(l, h) for l in range(38, 61) for h in range(mt.n_heads)]   # band_30_60
    return [(l, h) for l in range(30, 58) for h in range(mt.n_heads)]   # band_30_57
ABLATE_SET = _ablate_set()
ov  = OVReader(mt, READOUT_HEADS)
GSP = _build_causality_prompt(mt) if MODE == "causality" else None
FUNC.gsp = GSP
PREFIX = CAUSAL_PREFIX if MODE == "causality" else ""
rich_print(f"strategy={COLLAPSE_STRATEGY}  L_COL={L_COL}  collapse={len(COLLAPSE_HEADS)}h  "
           f"ablate={ABLATE_SET_NAME}({len(ABLATE_SET)}h)/{ABLATION_STRAT}  {DATATYPE} {SRC_LANG}->{DST_LANG}")

Building OV readers:   0%|          | 0/213 [00:00<?, ?it/s]

strategy=read_redirect  L_COL=36  collapse=213h  ablate=from_lcol(1120h)/blind_task  list python->python

## Collapse mechanisms (both validated)

`super_collapse_write` (Design B readout) reconstructs the summed goal-head OV write from `h^L_COL`
including Qwen2 q/k/v biases — matches a head's real OV at cosine 1.0. `super_layer_gen` injects that
write live during generation (accumulating an `h^L_COL` cache; KV-cache-safe). `live_collapse` is the
Design A surgical read-redirect.

In [4]:
# ── Design B: super-layer OV write (readout), with optional q/kv transfer from a SRC residual ──
def super_collapse_write(mt, goal_heads, h_read, pivot, *, transfer=None, src_resid_of=None,
                         dst_positions=None, src_positions=None, pivot_src=None):
    """Summed goal-head OV residual write at `pivot` when every goal head reads h_read (seq, n_embd).
    Real per-head attention (RoPE, causal) + o_proj, incl. Qwen2 q/k/v biases. Returns (n_embd,) f32.

    transfer:
      "kv" — recompute k/v at `dst_positions` from src_resid_of(L) at `src_positions` (SRC task span),
             placed at the DST positions (RoPE'd at the DST absolute position, like headwise_kv_patch).
      "q"  — recompute the pivot query from src_resid_of(L) at `pivot_src`.
    src_resid_of(L) returns the SRC residual for layer L (single h^L_COL, or per-layer native input)."""
    readers = _GoalReaders.build(mt, goal_heads)
    S = h_read.shape[0]; hd = readers.hd; group = readers.group
    acc = None
    for L, d in readers.by_layer.items():
        dev = d["Wq"].device
        rope = _apply_rope(mt, L); cos, sin = _rope_cos_sin(mt, range(S), device=dev)
        bq = baukit.get_module(mt._model, _proj(mt, L, "q")).bias
        bk = baukit.get_module(mt._model, _proj(mt, L, "k")).bias
        bv = baukit.get_module(mt._model, _proj(mt, L, "v")).bias
        x = _rmsnorm(h_read.to(dev), d["ln"]).to(d["Wq"].dtype)
        q_all = x @ d["Wq"].T + (bq if bq is not None else 0)
        k_all = x @ d["Wk"].T + (bk if bk is not None else 0)
        v_all = x @ d["Wv"].T + (bv if bv is not None else 0)
        if transfer in ("kv", "q") and src_resid_of is not None:
            xs = _rmsnorm(src_resid_of(L).to(dev), d["ln"]).to(d["Wq"].dtype)
            if transfer == "kv":
                k_src = xs @ d["Wk"].T + (bk if bk is not None else 0)
                v_src = xs @ d["Wv"].T + (bv if bv is not None else 0)
                for i, dp in enumerate(dst_positions):
                    k_all[dp] = k_src[src_positions[i]]; v_all[dp] = v_src[src_positions[i]]
            else:  # q
                q_src = xs @ d["Wq"].T + (bq if bq is not None else 0)
                q_all[pivot] = q_src[pivot_src]
        Wo = baukit.get_module(mt._model, _proj(mt, L, "o")).weight
        for j in d["js"]:
            kv = j // group
            q = q_all[:, j*hd:(j+1)*hd].unsqueeze(0).unsqueeze(0)
            k = k_all[:, kv*hd:(kv+1)*hd].unsqueeze(0).unsqueeze(0)
            v = v_all[:, kv*hd:(kv+1)*hd]
            q, k = rope(q, k, cos, sin)
            qp = q[0, 0, pivot]; kk = k[0, 0]
            scores = (kk @ qp) / (hd ** 0.5)
            scores = scores.masked_fill(torch.arange(S, device=scores.device) > pivot, float("-inf"))
            attn = torch.softmax(scores.float(), dim=-1).to(v.dtype)
            contrib = (Wo[:, j*hd:(j+1)*hd] @ (attn @ v)).float().cpu()
            acc = contrib if acc is None else acc + contrib
    return acc

def verbalize_write(vec):
    """GoalScope-style readout of a raw aggregated head-write vector (additive_norm into the probe)."""
    out = head_additive_generate(mt, ov.probe_inputs(MODE), vec.to(mt.device),
                                 layers=list(range(*STEER_BAND)), max_new_tokens=MAXNEW,
                                 strategy="additive_norm", alpha=ALPHA_OV)
    return out if isinstance(out, str) else out[0]

# ── Design B: super-layer injection during generation (live, cache-safe), + optional q/kv transfer ──
@contextmanager
def super_layer_gen(mt, l_col, goal_heads, *, transfer=None, src_resid_of=None,
                    dst_positions=None, src_positions=None, pivot_dst=None, pivot_src=None):
    if not goal_heads:
        yield; return
    readers = _GoalReaders.build(mt, goal_heads); hd, group = readers.hd, readers.group
    meta = {}
    for L, d in readers.by_layer.items():
        g = lambda w: baukit.get_module(mt._model, _proj(mt, L, w))
        meta[L] = dict(d=d, bq=g("q").bias, bk=g("k").bias, bv=g("v").bias, Wo=g("o").weight, rope=_apply_rope(mt, L))
    st = {"resid": None}; handles = []
    def block_hook(module, args, output):
        out = output[0] if isinstance(output, tuple) else output      # (B,S,E) h^{l_col} current
        B, S, E = out.shape
        full = out.detach() if st["resid"] is None else torch.cat([st["resid"].to(out.device), out.detach()], dim=1)
        st["resid"] = full; T = full.shape[1]; qs = T - S
        add = torch.zeros(B, S, E, device=out.device, dtype=torch.float32)
        for L, mm in meta.items():
            d = mm["d"]; dev = d["Wq"].device
            cos, sin = _rope_cos_sin(mt, range(T), device=dev)
            x = _rmsnorm(full.to(dev), d["ln"]).to(d["Wq"].dtype)
            q_all = x @ d["Wq"].T + (mm["bq"] if mm["bq"] is not None else 0)
            k_all = x @ d["Wk"].T + (mm["bk"] if mm["bk"] is not None else 0)
            v_all = x @ d["Wv"].T + (mm["bv"] if mm["bv"] is not None else 0)
            if transfer in ("kv", "q") and src_resid_of is not None:
                xs = _rmsnorm(src_resid_of(L).to(dev), d["ln"]).to(d["Wq"].dtype)
                if transfer == "kv":
                    k_src = xs @ d["Wk"].T + (mm["bk"] if mm["bk"] is not None else 0)
                    v_src = xs @ d["Wv"].T + (mm["bv"] if mm["bv"] is not None else 0)
                    for i, dp in enumerate(dst_positions):
                        k_all[:, dp] = k_src[src_positions[i]]; v_all[:, dp] = v_src[src_positions[i]]
                else:  # q — swap the pivot query (only affects the prefill forward where pivot is a query)
                    q_src = xs @ d["Wq"].T + (mm["bq"] if mm["bq"] is not None else 0)
                    q_all[:, pivot_dst] = q_src[pivot_src]
            for j in d["js"]:
                kv = j // group
                q = q_all[..., j*hd:(j+1)*hd].unsqueeze(1); k = k_all[..., kv*hd:(kv+1)*hd].unsqueeze(1)
                v = v_all[..., kv*hd:(kv+1)*hd]
                q, k = mm["rope"](q, k, cos, sin)
                qc = q[:, 0, qs:T, :]; kk = k[:, 0, :, :]
                scores = torch.einsum("bsh,bth->bst", qc.float(), kk.float()) / (hd ** 0.5)
                qi = torch.arange(qs, T, device=scores.device).view(1, S, 1); ki = torch.arange(T, device=scores.device).view(1, 1, T)
                attn = torch.softmax(scores.masked_fill(ki > qi, float("-inf")), dim=-1).to(v.dtype)
                head_out = torch.einsum("bst,bth->bsh", attn, v)
                add += torch.einsum("eh,bsh->bse", mm["Wo"][:, j*hd:(j+1)*hd].float(), head_out.float()).to(add.device)
        new = out + add.to(out.dtype)
        return (new,) + tuple(output[1:]) if isinstance(output, tuple) else new
    try:
        handles.append(baukit.get_module(mt._model, mt.layer_name_format.format(l_col)).register_forward_hook(block_hook))
        yield
    finally:
        for h in handles: h.remove()

# ── Design A: surgical read-redirect (goal heads l>L_COL recompute q/k/v from h^L_COL) ──
@contextmanager
def live_collapse(mt, l_col, collapse_heads):
    heads = [(l, h) for (l, h) in collapse_heads if l > l_col]
    if not heads:
        yield; return
    readers = _GoalReaders.build(mt, heads); holder = {}; handles = []
    def cap(module, args, output):
        holder["h"] = (output[0] if isinstance(output, tuple) else output).detach(); return None
    def mk(L, which):
        d = readers.by_layer[L]; hd = readers.hd; group = readers.group
        W = d["Wq"] if which == "q" else (d["Wk"] if which == "k" else d["Wv"])
        b = baukit.get_module(mt._model, _proj(mt, L, which)).bias
        def hook(module, args, output):
            h = holder.get("h")
            if h is None: return None
            new = (_rmsnorm(h.to(W.device), d["ln"]) @ W.T) + (b if b is not None else 0)
            out = output.clone()
            for j in d["js"]:
                sl = slice(j*hd, (j+1)*hd) if which == "q" else slice((j//group)*hd, (j//group+1)*hd)
                out[..., sl] = new[..., sl].to(out.dtype).to(out.device)
            return out
        return hook
    try:
        handles.append(baukit.get_module(mt._model, mt.layer_name_format.format(l_col)).register_forward_hook(cap))
        for L in sorted({l for l, _ in heads}):
            for which in ("q", "k", "v"):
                handles.append(baukit.get_module(mt._model, _proj(mt, L, which)).register_forward_hook(mk(L, which)))
        yield
    finally:
        for h in handles: h.remove()

In [5]:
# ── data helpers (from 07.3) ──
BANK = DATA.QTASKS[DATATYPE]; BY = {p.id: p for p in BANK.predicates}
A, B, C, D, E = "total_length", "total_uppercase_letters", "has_palindrome", "all_unique_lengths", "count_starts_vowel"

def mk(pred_ids, progress, target, lang="python"):
    task = DATA.dataclasses.replace(BANK, id="xtask", predicates=tuple(BY[i] for i in pred_ids))
    return DATA.QTransferInstance(task=task, var_name=BANK.var_names[0], presentation="listed",
        src_lang=lang, dst_lang=lang, src_progress=progress, dst_progress=progress,
        expected_next=target, naming="letters", meta={"datatype": DATATYPE})

def list_body(inst):
    return "\n".join(f"({i+1}) {p.description}" for i, p in enumerate(inst.task.predicates))

def get_prompt(question, prefill, ctx=None):
    return FUNC.get_prompt(question.replace("computes:", "computes in sequential order:"), prefill, ctx=ctx)

def stub_parts(inst, side, stub):
    q, pf = (inst.src_parts() if side == "src" else inst.dst_parts())
    if stub != "var_eq":
        return q, pf
    lang = inst.src_lang if side == "src" else inst.dst_lang
    prog = inst.src_progress if side == "src" else inst.dst_progress
    var = inst.var_of(inst.expected_next); body = pf.rstrip("\n ")
    if lang == "cpp":
        for pid in prog: body = DATA._cpp_autoify(body, inst.var_of(pid))
        return q, body + f"\n    auto {var} ="
    return q, body + f"\n    {var} ="

def ov_boost(positions=(-1,)):
    return boost_head_ov(mt, PATCH_HEADS, OV_BOOST, positions=positions) if OV_BOOST != 1.0 else nullcontext()

def show_spans(tok, title, task_span=None, prog_span=None, piv=-1):
    ids = (prepare_input(prompts=tok, tokenizer=mt) if isinstance(tok, str) else tok)["input_ids"][0].tolist()
    n = len(ids); pivpos = n + piv if piv < 0 else piv
    task = set(range(*task_span)) if task_span else set(); prog = set(range(*prog_span)) if prog_span else set()
    PO, PC, KO, KC, TO, TC = "\x00<", "\x00>", "\x01<", "\x01>", "\x02<", "\x02>"
    parts = []
    for i in range(n):
        s = mt.tokenizer.decode([ids[i]]).replace("\n", "\\n\n")
        if i == pivpos:   s = PO + s + PC
        elif i in prog:   s = KO + s + KC
        elif i in task:   s = TO + s + TC
        parts.append(s)
    txt = escape("".join(parts))
    txt = (txt.replace(PO, "[bold black on salmon1]").replace(PC, "[/bold black on salmon1]")
              .replace(KO, "[bold black on dark_orange]").replace(KC, "[/bold black on dark_orange]")
              .replace(TO, "[bright_yellow]").replace(TC, "[/bright_yellow]"))
    rich_print(Panel(Text.from_markup(txt), title=f"[bold]{escape(title)}[/bold]", title_align="left",
                     border_style="grey42", padding=(1, 1)))

In [6]:
# ── contrastive pair: shared progress, different task (SRC target != DST frontier) ──
# k_src = mk([A, B, C], [A, B], C)
# k_dst = mk([B, D, A], [B, A], D)

k_src = mk([A, B, C, D], [A, B], C)
k_dst = mk([A, B, D, C], [A], B)

# k_src, k_dst = k_dst, k_src
k_src.src_lang = SRC_LANG; k_dst.dst_lang = DST_LANG
_q, _pf = stub_parts(k_src, "src", KV_STUB); ksrc_prompt = get_prompt(_q, DATA._finalize_prefill(_pf))
_q, _pf = stub_parts(k_dst, "dst", KV_STUB); kdst_prompt = get_prompt(_q, DATA._finalize_prefill(_pf))
src_text, dst_text = list_body(k_src), list_body(k_dst)
src_span0 = token_span(ksrc_prompt, src_text); dst_span0 = token_span(kdst_prompt, dst_text)
ksrc_tok, kdst_tok, src_pos, dst_pos, src_span, dst_span = align_kv_spans(
    mt, ksrc_prompt, kdst_prompt, src_span0, dst_span0, mode="pad", src_text=src_text, dst_text=dst_text)
kpiv_src = ksrc_tok["input_ids"].shape[1] - 1; kpiv_dst = kdst_tok["input_ids"].shape[1] - 1
SRC_TARGET = k_src.expected_next; DST_NATIVE = k_dst.expected_next
rich_print(f"SRC target {SRC_TARGET} = [green]{escape(BY[SRC_TARGET].description)}[/]   "
           f"DST native {DST_NATIVE} = [magenta]{escape(BY[DST_NATIVE].description)}[/]")
rich_print(Panel(escape(ksrc_prompt), title="[bold]FULL SOURCE PROMPT[/bold]", title_align="left",
                 border_style="green", padding=(1, 1)))
rich_print(Panel(escape(kdst_prompt), title="[bold]FULL DEST PROMPT[/bold]", title_align="left",
                 border_style="magenta", padding=(1, 1)))

SRC target has_palindrome = whether any word is a palindrome   DST native total_uppercase_letters = the total 
number of uppercase letters across all words

╭─ FULL SOURCE PROMPT ────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes in    │
│ sequential order:                                                                                               │
│ (1) the total number of characters across all words                                                             │
│ (2) the total number of uppercase letters across all words                                                      │
│ (3) whether any word is a palindrome                                                                            │
│ (4) whether all words have distinct lengths                                                                     │
│ <|im_end|>                                                                                                      │
│ <|im_start|>assistant                                                                                           │
│ <code>                                                                                                          │
│ def func(words: list[str]):                                                                                     │
│     a = sum(len(w) for w in words)                                                                              │
│     b = sum(1 for w in words for c in w if c.isupper())                                                         │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ FULL DEST PROMPT ──────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes in    │
│ sequential order:                                                                                               │
│ (1) the total number of characters across all words                                                             │
│ (2) the total number of uppercase letters across all words                                                      │
│ (3) whether all words have distinct lengths                                                                     │
│ (4) whether any word is a palindrome                                                                            │
│ <|im_end|>                                                                                                      │
│ <|im_start|>assistant                                                                                           │
│ <code>                                                                                                          │
│ def func(words: list[str]):                                                                                     │
│     a = sum(len(w) for w in words)                                                                              │
│     b =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Ablation spans + run stacks

`ABLATION_STRAT`: **blind_task** (task span) · **blind_task_and_progress** (task→progress, keep last-3) ·
**sink_to_pivot** (all but last-3). Under **super_layer** the ablation includes the native goal heads
(`keep=None`), so the goal is expressed only by the injected super-layer; under **read_redirect** the
collapsed heads are kept (`keep=COLLAPSE_HEADS`).

In [7]:
def _block_positions(side, strat):
    span, P_ = (src_span, ksrc_tok["input_ids"].shape[1]) if side == "src" else (dst_span, kdst_tok["input_ids"].shape[1])
    keep_from = P_ - LAST_K_KEEP
    if strat == "off":                     return []
    if strat == "blind_task":              return list(range(span[0], span[1]))
    if strat == "blind_task_and_progress": return list(range(span[0], keep_from))
    if strat == "sink_to_pivot":           return list(range(0, keep_from))
    raise ValueError(strat)

def ablate_ctx(side):
    if ABLATION_STRAT == "off":
        return nullcontext()
    block = _block_positions(side, ABLATION_STRAT)
    keep = COLLAPSE_HEADS if (KEEP_COLLAPSED and COLLAPSE_STRATEGY == "read_redirect") else None
    return restrict_head_attn(mt, ABLATE_SET, block, keep=keep)

_blk = _block_positions("dst", ABLATION_STRAT)
show_spans(kdst_tok, f"DST · task {tuple(dst_span)} · ablate[{ABLATION_STRAT}] {len(_blk)} keys · pivot@{kpiv_dst}",
           task_span=dst_span, prog_span=(min(_blk), max(_blk) + 1) if _blk else None, piv=-1)

# ── transfer caches (read_redirect transfer runs) ──
kvheads = heads_per_layer(PATCH_HEADS, "k", layers=set(FUNC._layers_of(PATCH_HEADS)))
qh      = heads_per_layer(PATCH_HEADS, "q")
def _cache(which, positions):
    ctx = live_collapse(mt, L_COL, COLLAPSE_HEADS) if TRANSFER_SRC == "collapsed_clean" else nullcontext()
    with ctx:
        heads = sorted(kvheads) if which in ("k", "v") else sorted({l for l, _ in PATCH_HEADS})
        return cache_src_proj(ksrc_tok, positions, heads, which)
k_cache = _cache("k", src_pos); v_cache = _cache("v", src_pos); q_cache = _cache("q", [kpiv_src])
q_specs = [dict(which="q", cache=q_cache, heads=qh, positions=[kpiv_dst])]

# ── super_layer transfer source: SRC residual the super-layer sources k/v (or q) from ──
#   collapsed_clean -> single SRC h^L_COL ; clean -> per-layer native SRC input h^{l-1}
SRC_RESID_OF = None
if COLLAPSE_STRATEGY == "super_layer":
    _H_SRC_COL = resid_all(mt, ksrc_tok, L_COL)
    if TRANSFER_SRC == "collapsed_clean":
        SRC_RESID_OF = lambda L: _H_SRC_COL
    else:  # clean — SRC residual at each goal-head layer's native input, one forward
        _gl = sorted({l for l, _ in COLLAPSE_HEADS})
        _names = {l: mt.layer_name_format.format(l - 1) for l in _gl}
        with baukit.TraceDict(mt._model, list(_names.values()), retain_output=True) as _tr:
            mt._model(input_ids=ksrc_tok["input_ids"], attention_mask=ksrc_tok["attention_mask"])
        _SRCN = {}
        for l in _gl:
            _o = _tr[_names[l]].output; _o = _o[0] if isinstance(_o, tuple) else _o
            _SRCN[l] = _o[0].detach().float().cpu()
        SRC_RESID_OF = lambda L: _SRCN[L]

def gen_ctx(side, transfer=None):
    """Collapse + ablation (+ transfer), held across all decode steps."""
    es = ExitStack()
    if COLLAPSE_STRATEGY == "super_layer":
        es.enter_context(super_layer_gen(mt, L_COL, COLLAPSE_HEADS, transfer=transfer,
                                         src_resid_of=(SRC_RESID_OF if transfer else None),
                                         dst_positions=dst_pos, src_positions=src_pos,
                                         pivot_dst=kpiv_dst, pivot_src=kpiv_src))
        es.enter_context(ablate_ctx(side))
    else:
        es.enter_context(live_collapse(mt, L_COL, COLLAPSE_HEADS))
        es.enter_context(ablate_ctx(side))
        if transfer == "kv":
            es.enter_context(headwise_kv_patch(mt, PATCH_HEADS, k_cache, v_cache, dst_pos)); es.enter_context(ov_boost())
        elif transfer == "q":
            es.enter_context(proj_patch_hooks(mt, q_specs)); es.enter_context(ov_boost())
    return es

def _iface(stack_fn):
    def _if(mt, inputs, patches=None, model_kwargs=None):
        with stack_fn():
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return _if

def _ov_layerwise(tok, pivot, stack_fn):
    ln = lambda l: mt.layer_name_format.format(l); unt = lambda x: x[0] if isinstance(x, tuple) else x
    ctx = stack_fn() if stack_fn is not None else nullcontext()
    with torch.no_grad(), ctx, baukit.TraceDict(mt._model, [ln(l) for l in LAYERS], retain_output=True) as tr:
        mt._model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"])
    resid = {l: unt(tr[ln(l)].output)[0, pivot].detach().float().cpu() for l in LAYERS}
    outs = ov.apply_probe(resid, mode=MODE, steer_layers=list(range(*STEER_BAND)), max_new_tokens=MAXNEW, alpha=ALPHA_OV)
    for l in LAYERS: print(f"    OVReader L={l:2d}: {escape(PREFIX + outs[l])}")

╭─ DST · task (66, 110) · ablate[blind_task] 44 keys · pivot@140 ─────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes in    │
│ sequential order:\n                                                                                             │
│ (1) the total number of characters across all words\n                                                           │
│ (2) the total number of uppercase letters across all words\n                                                    │
│ (3) whether all words have distinct lengths\n                                                                   │
│ (4) whether any word is a palindrome\n                                                                          │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(len(w) for w in words)\n                                                                            │
│     b =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
def run(tag, tok, pivot, side=None, transfer=None, ov_on=False, collapsed=True):
    """Full generation (intervention held throughout) + readout @ pivot (+ optional OVReader)."""
    if GEN_ON:
        ctx = gen_ctx(side, transfer) if collapsed else nullcontext()
        with ctx:
            g = greedy(tok, max_new=GEN_NEW)
        rich_print(f"[dim]{tag} · generation:[/]\n{escape(g)}")
    if not collapsed:
        goalscope_readout(f"  goalscope · {tag} @ pivot", tok, pivot, heads=READOUT_HEADS)
        if ov_on: _ov_layerwise(tok, pivot, None)
    elif COLLAPSE_STRATEGY == "super_layer":
        kw = {}
        if transfer == "kv":
            kw = dict(transfer="kv", src_resid_of=SRC_RESID_OF, dst_positions=dst_pos, src_positions=src_pos)
        elif transfer == "q":
            kw = dict(transfer="q", src_resid_of=SRC_RESID_OF, pivot_src=kpiv_src)
        vec = super_collapse_write(mt, COLLAPSE_HEADS, resid_all(mt, tok, L_COL), pivot, **kw)
        rich_print(f"  goalscope[super-layer] · {tag} @ pivot: [italic]{escape(verbalize_write(vec))}[/]")
        if ov_on: _ov_layerwise(tok, pivot, lambda: gen_ctx(side, transfer))
    else:  # read_redirect
        goalscope_readout(f"  goalscope · {tag} @ pivot", tok, pivot, heads=READOUT_HEADS,
                          iface=_iface(lambda: gen_ctx(side, transfer)))
        if ov_on: _ov_layerwise(tok, pivot, lambda: gen_ctx(side, transfer))

## The four runs (full generation + readout, same intervention throughout)

In [9]:
# ── RUN 1 · clean (SRC & DST) ──
rich_print(f"[bold]=== RUN 1 · CLEAN ===[/]  SRC target {SRC_TARGET}={BY[SRC_TARGET].description!r} · "
           f"DST native {DST_NATIVE}={BY[DST_NATIVE].description!r}")
run("clean SRC", ksrc_tok, kpiv_src, collapsed=False, ov_on=OV_CLEAN)
run("clean DST", kdst_tok, kpiv_dst, collapsed=False, ov_on=OV_CLEAN)

=== RUN 1 · CLEAN ===  SRC target has_palindrome='whether any word is a palindrome' · DST native 
total_uppercase_letters='the total number of uppercase letters across all words'

clean SRC · generation:
 any(w == w[::-1] for w in words)
    d = len(set(len(w) for w in words)) == len(words)
    return a, b, c, d
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


  goalscope · clean SRC @ pivot  pos=159 ' =' => I'm checking if any string in the list is a palindrome by using a 
generator expression: `any(s == s[::-1] for s in strings)`.

clean DST · generation:
 sum(1 for w in words for c in w if c.isupper())
    c = len(set(len(w) for w in words)) == len(words)
    d = any(w == w[::-1] for w in words)
    return a, b, c, d
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

  goalscope · clean DST @ pivot  pos=140 ' =' => Counting uppercase letters.

In [10]:
# ── RUN 2 · collapsed_clean (SRC & DST) ──
rich_print(f"[bold]=== RUN 2 · COLLAPSED_CLEAN ({COLLAPSE_STRATEGY}) ===[/]  "
           f"(L_COL={L_COL}, ablate={ABLATE_SET_NAME}/{ABLATION_STRAT})")
run("collapsed SRC", ksrc_tok, kpiv_src, side="src", ov_on=OV_COLLAPSED)
run("collapsed DST", kdst_tok, kpiv_dst, side="dst", ov_on=OV_COLLAPSED)

=== RUN 2 · COLLAPSED_CLEAN (read_redirect) ===  (L_COL=36, ablate=from_lcol/blind_task)

collapsed SRC · generation:
 all(len(w) == len(set(w)) for w in words)
    return a, b, c
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

  goalscope · collapsed SRC @ pivot  pos=159 ' =' => I'm checking if the current line contains any unique 
characters.

    OVReader L=38: Sure, I can help with that. Please provide the list of elements you want to check for uniqueness using a set comprehension. Here's how you can do it:
```python
all_unique = len(set(lst) for lst in \[set(x) for x in lst] if x == 1)
```
This
    OVReader L=40: It seems like you're asking if all elements in a list are unique by comparing the length of the list with the length of the set created from the list. If the length of the set (which removes duplicates) is equal to the length of the list, it means all elements are unique. Here's a concise way
    OVReader L=42: It seems like you're asking if all elements in a list are unique by checking if the length of the list is equal to the length of the set of the list when converted to a set of booleans indicating whether all elements are unique. This can be done using a simple check with the `len()` function on a
    OVReader L=44: I'm checking if all elements in a list are unique by converting the list to a set and compar

collapsed DST · generation:
 all(len(w) == len(set(w)) for w in words)
    c = any(w == w[::-1] for w in words)
    return a, b, c
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

  goalscope · collapsed DST @ pivot  pos=140 ' =' => I'm analyzing the specific characters and operations within 
the string to count the total number of characters.

    OVReader L=38: Counting unique characters by using a set to find distinct elements.
    OVReader L=40: I'm checking if all characters in the string are unique by using a set to count distinct characters.
    OVReader L=42: I'm checking if all characters in the string are unique by comparing the length of the string with the length of the set that contains only unique elements. This checks for case-insensitive uniqueness by converting each character to lowercase and summing up the boolean value of the set's size, which counts the number of unique elements when
    OVReader L=44: I'm not sure what specific criteria you're using to measure uniqueness, but if you're looking for a way to determine the number of unique elements by comparing case sensitivity, you can use a set to track characters with case sensitivity and insensitivity. Here's a quick way to count unique elements:

```python
unique_elements
    OVReader L=46: I'm counting characters and checking for case sensitivity.
    

In [11]:
# ── RUN 3 · collapsed KV-transfer: collapsed DST goal heads read k/v from SRC task span (q native) ──
rich_print(f"[bold]=== RUN 3 · COLLAPSED KV-TRANSFER ({COLLAPSE_STRATEGY}) ===[/]  expect SRC {SRC_TARGET}="
           f"{BY[SRC_TARGET].description!r}  (transfer_src={TRANSFER_SRC})")
run("collapsed kv DST", kdst_tok, kpiv_dst, side="dst", transfer="kv", ov_on=OV_KV)

=== RUN 3 · COLLAPSED KV-TRANSFER (read_redirect) ===  expect SRC has_palindrome='whether any word is a palindrome'
(transfer_src=collapsed_clean)

collapsed kv DST · generation:
 any(w == w[::-1] for w in words)
    return a, b
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

  goalscope · collapsed kv DST @ pivot  pos=140 ' =' => I'm analyzing the specific line of code to determine what 
actions or operations it performs, such as counting characters, checking for specific letters, or identifying any 
unique properties related to the text.

    OVReader L=38: Counting unique characters by using a set to find distinct elements.
    OVReader L=40: I'm checking if a character is uppercase and not repeated. It counts characters that appear only once.
    OVReader L=42: I'm checking if a variable is uppercase by comparing it to its lowercase version. The comparison checks for characters that have different case (uppercase) by using the `sum` function with a generator expression that counts characters that are uppercase.
    OVReader L=44: I'm not sure what specific criteria you're using to measure "briefness," but I can provide a briefness score for strings by summing up the ASCII value of each character, treating uppercase letters as lowercase by converting them to lowercase and then summing their values. Here's how you can calculate the briefness score
    OVReader L=46: I'm currently tracking the number of uppercase letters that have been converted to lowercase due to case folding for ASCII characters.
    OVReader L=48: I'

In [ ]:
# ── RUN 4 · collapsed q-transfer: collapsed DST goal-head q patched from SRC (k/v native) ──
rich_print(f"[bold]=== RUN 4 · COLLAPSED Q-TRANSFER ({COLLAPSE_STRATEGY}) ===[/]  expect SRC {SRC_TARGET}="
           f"{BY[SRC_TARGET].description!r}  (transfer_src={TRANSFER_SRC})")
run("collapsed q DST", kdst_tok, kpiv_dst, side="dst", transfer="q", ov_on=OV_QT)

=== RUN 4 · COLLAPSED Q-TRANSFER (read_redirect) ===  expect SRC has_palindrome='whether any word is a palindrome' 
(transfer_src=collapsed_clean)

collapsed q DST · generation:
 len(set(words))
    c = all(len(w) == len(set(w)) for w in words)
    d = any(w == w[::-1] for w in words)
    return a, b, c, d
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

  goalscope · collapsed q DST @ pivot  pos=140 ' =' => I'm checking if the current line of code is unique or not, 
by comparing it with all previous lines to see if there are any duplicates.

    OVReader L=38: Counting unique characters by using a set to filter out duplicates.
    OVReader L=40: I'm checking if all elements in the list are unique by comparing the length of the list with the length of the set created from the list (which removes duplicates). If the length of the list is equal to the length of the set of the list, it means all elements are unique.
    OVReader L=42: I'm checking if all elements in the list are unique by comparing the length of the list with the length of the set of the list, which checks for uniqueness by ensuring that the number of elements is equal to the number of unique elements when converted to a set. This confirms that all elements are distinct.
    OVReader L=44: I'm checking if all elements in a list are unique by comparing the length of the list with the length of the set of the list. If the lengths are equal, it means there are no duplicate elements.
    OVReader L=46: I'm checking if all elements in a list are unique by comparing

: 

## Summary

Per run, read the DST **generation** and the readout @ pivot: SRC's task (transfer), DST's frontier
(reverted), or hybrid. **super_layer** (Design B) expresses the goal exactly once — the summed goal-head
OV from `h^L_COL`, with the native heads ablated — so the readout/generation reflect a single collapsed
depth with no double-counting. **read_redirect** (Design A) keeps the collapsed heads live and reads them.
Transfer works under both strategies: **KV-transfer** vs **q-transfer** localizes whether goal selection
rides the task-span content or the pivot query; `TRANSFER_SRC=clean` vs `collapsed_clean` tests whether
native-depth vs collapsed-depth SRC q/kv carry the same content.